In [ ]:
# ======================================================================
# MODULO A — Configurazione e Parametri
# A.1 — Definizione dei parametri globali del modello e della pipeline
# ======================================================================


# === MODALITA' DI ESECUZIONE ===
# True = Google Colab (default), False = esecuzione in locale
COLAB = True

# --- Import condizionali ---
if COLAB:
    from google.colab import files

import os

# --- Percorsi di output ---
if COLAB:
    OUTPUT_DIR = "/content/output/"
else:
    OUTPUT_DIR = "./output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Salvataggio file di output in locale ---
# Rilevante solo quando COLAB=False:
# False = salva in ./output/ (default), True = apre finestra di dialogo per scegliere il percorso
ASK_SAVE_PATH = False

VERBOSE = False  # se True mostra log dettagliati durante l'esecuzione

# --- Mappatura colonne del file Excel di input ---
ID_COL        = "SKU"                    # chiave prodotto (codice univoco)
DESC_COL      = "Description"            # descrizione prodotto
LT_COL_NAME   = "LT"                    # lead time (giorni)
PACK_SIZE_COL = "Round"                  # multiplo d'imballo
UOM_COL       = "BUn"                    # unita' di misura

# === FORECAST FUTURO ===
HORIZON = 25                             # mesi da prevedere
TRIM_LEADING_ZEROS = True                # ignora zeri iniziali (prodotto non ancora lanciato)
MIN_HISTORY_POINTS = 6                   # minimo punti storici richiesti per SKU

# === BACKTEST ===
HORIZON_BACKTEST = 12                    # mesi di backtest (finestra di valutazione)
# Griglia di scaling factor per ottimizzare (passo 0.05 per maggiore precisione)
QUANTILE_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
                 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
N_BACKTEST_ORIGINS   = 2                 # origini backtest (1 = originale, 2+ = rolling-origin)
SHRINKAGE_ENABLED    = True              # shrinkage scaling factor verso mediana globale

# === OUTLIER (winsorizing sulla serie storica) ===
REMOVE_OUTLIERS = True                   # abilita/disabilita filtro
OUTLIER_LEVEL   = 0.05                   # clip al 5 e 95 percentile

# === ARROTONDAMENTO ===
ROUNDING_MODE    = "nearest"             # "up" | "down" | "nearest"
ROUND_DECIMALS   = 3                     # decimali dopo arrotondamento

# === CALIBRAZIONE STAGIONALE ===
# Mesi su cui applicare l'aggiustamento stagionale (es. agosto, dicembre)
CALIBRATION_MONTHS = [8, 12]             # [] = calibrazione disattivata

# === INVENTARIO E SCORTA DI SICUREZZA ===
CALCULATE_SS       = True                # attiva/disattiva calcolo scorta di sicurezza
DEFAULT_LEAD_TIME  = 30                  # giorni (fallback se colonna LT manca)
REORDER_PERIOD     = 30                  # giorni (1 mese fisso come da richiesta)
SS_LOOKBACK_MONTHS = 12                  # mesi storici per calcolo deviazione standard

# Soglie ABC (classificazione Pareto sui volumi cumulativi)
ABC_LIMITS = {
    "A": 0.70,  # primi 70% del volume
    "B": 0.90,  # dal 70% al 90%
    "C": 1.00   # resto
}

# Soglie XYZ (coefficiente di variazione CV = DevStd / Media)
XYZ_LIMITS = {
    "X": 0.40,  # CV <= 0.4 (domanda molto stabile)
    "Y": 0.80,  # 0.4 < CV <= 0.8 (domanda variabile)
    "Z": 999.0  # CV > 0.8 (domanda erratica)
}

# Livelli di servizio target per classe ABC/XYZ
SERVICE_LEVEL_MATRIX = {
    "AX": 0.97, "AY": 0.95, "AZ": 0.93,
    "BX": 0.91, "BY": 0.90, "BZ": 0.89,
    "CX": 0.87, "CY": 0.80, "CZ": 0.00   # 0.00 = nessuna scorta di sicurezza
}

# === OUTPUT ===
from datetime import datetime
OUTPUT_FORMAT     = "xlsx"
OUTPUT_FILE_BASE  = "Forecast and SS"
OUTPUT_SUFFIX     = datetime.now().strftime(" %Y %m %d %H_%M")  # suffisso: data e ora corrente

# === TIMER AUTOMATICO (misura il tempo di esecuzione di ogni cella) ===
import time as _time
from IPython import get_ipython as _get_ipython

_cell_times = {}
_cell_start = {}
_suppress_timer_print = False

def _timer_start(info=None):
    _cell_start['t'] = _time.time()

def _timer_stop(info=None):
    global _suppress_timer_print
    elapsed = _time.time() - _cell_start.get('t', _time.time())
    shell = _get_ipython()
    cell_num = shell.execution_count
    _cell_times[cell_num] = elapsed
    if _suppress_timer_print:
        _suppress_timer_print = False
        return
    minuti, secondi = divmod(elapsed, 60)
    if minuti >= 1:
        print(f"\u23f1\ufe0f Cella [{cell_num}]: {int(minuti)}m {secondi:.1f}s")
    else:
        print(f"\u23f1\ufe0f Cella [{cell_num}]: {secondi:.1f}s")

_ip = _get_ipython()
# Rimuovi eventuali registrazioni precedenti per evitare duplicati
for _cb in list(_ip.events.callbacks.get('pre_run_cell', [])):
    if _cb.__name__ == '_timer_start':
        _ip.events.unregister('pre_run_cell', _cb)
for _cb in list(_ip.events.callbacks.get('post_run_cell', [])):
    if _cb.__name__ == '_timer_stop':
        _ip.events.unregister('post_run_cell', _cb)
_ip.events.register('pre_run_cell', _timer_start)
_ip.events.register('post_run_cell', _timer_stop)

In [ ]:
# ======================================================================
# MODULO A — Configurazione e Parametri
# A.2 — Import delle librerie principali
# ======================================================================

import pandas as pd
import numpy as np
import re
import math

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.1 — Caricamento file Excel (Colab: upload / Locale: finestra di dialogo)
# ======================================================================

import pandas as pd

if COLAB:
    # --- Modalita' Colab: upload tramite popup del browser ---
    from google.colab import files
    print("📥 Carica il file Excel dal tuo PC...")
    uploaded = files.upload()  # apre la finestra di selezione file
    INPUT_FILE = list(uploaded.keys())[0]
else:
    # --- Modalita' locale: finestra di dialogo del sistema operativo ---
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()  # nasconde la finestra principale di tkinter
    root.attributes('-topmost', True)  # forza il dialog in primo piano su Windows
    INPUT_FILE = filedialog.askopenfilename(
        title="Seleziona il file Excel di input",
        filetypes=[("Excel files", "*.xlsx *.xls")],
        parent=root
    )
    root.destroy()
    if not INPUT_FILE:
        raise ValueError("❌ Nessun file selezionato. Operazione annullata.")
    print(f"📥 File selezionato: {INPUT_FILE}")

# Legge il primo foglio disponibile
all_sheets = pd.read_excel(INPUT_FILE, sheet_name=None)
first_sheet_name = list(all_sheets.keys())[0]
df_raw = all_sheets[first_sheet_name]

print("✔️ Foglio caricato:", first_sheet_name)
df_raw.head()

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.2 — Identificazione automatica delle colonne temporali (YYYY_MM)
# ======================================================================

date_col_pattern = re.compile(r"^\d{4}_\d{2}$")

# Separa colonne temporali (domanda mensile) da colonne metadati
date_cols = [col for col in df_raw.columns if date_col_pattern.match(str(col))]
meta_cols = [col for col in df_raw.columns if col not in date_cols]

print("Colonne temporali trovate:", len(date_cols))
print(date_cols[:10], "...")
print("\nColonne metadati:", meta_cols)

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.3 — Sostituzione valori mancanti (NaN → 0) nelle colonne temporali
# ======================================================================

print("➡️ Conversione NaN delle colonne temporali in 0...")
df_raw[date_cols] = df_raw[date_cols].fillna(0)
print("✔️ NaN convertiti in 0.")

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.4 — Conversione formato WIDE → LONG (una riga per SKU/mese)
# ======================================================================

# Trasforma da formato wide (una colonna per mese) a long (una riga per osservazione)
df_long = df_raw.melt(
    id_vars=meta_cols,
    value_vars=date_cols,
    var_name="Period",
    value_name="Demand"
)

df_long = df_long.dropna(subset=["Demand"]).reset_index(drop=True)

print(df_long.head())
print("\nShape finale:", df_long.shape)

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.5 — Parsing del periodo → datetime e ordinamento cronologico
# ======================================================================

# Converte la stringa "YYYY_MM" in data e ordina per SKU e mese
df_long["Date"] = pd.to_datetime(df_long["Period"], format="%Y_%m")
df_long = df_long.sort_values(by=[ID_COL, "Date"]).reset_index(drop=True)

print(df_long[[ID_COL, "Period", "Date", "Demand"]].head(10))

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.6 — Filtro SKU con storico insufficiente
# ======================================================================

# Conta i mesi di storico per ogni SKU
hist_counts = (
    df_long.groupby(ID_COL)["Demand"]
    .count()
    .reset_index(name="history_points")
)

# Mantiene solo gli SKU con almeno MIN_HISTORY_POINTS osservazioni
valid_skus = hist_counts[hist_counts["history_points"] >= MIN_HISTORY_POINTS][ID_COL]

print(f"SKU totali: {df_long[ID_COL].nunique()}")
print(f"SKU con storico >= {MIN_HISTORY_POINTS}: {len(valid_skus)}")

df_filtered = (
    df_long[df_long[ID_COL].isin(valid_skus)]
    .reset_index(drop=True)
    .copy()
)

print("\nShape finale dati filtrati:", df_filtered.shape)
df_filtered.head(10)

In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.7 — Winsorizing degli outlier per SKU
# ======================================================================

def winsorize_series(s, level=OUTLIER_LEVEL):
    """
    Applica winsorizing a una serie: taglia i valori sotto il percentile
    `level` e sopra il percentile `1-level`.
    Se REMOVE_OUTLIERS è False, restituisce la serie invariata.
    """
    if not REMOVE_OUTLIERS:
        return s

    vals = s.dropna()
    if len(vals) == 0:
        return s

    q_low = vals.quantile(level)
    q_high = vals.quantile(1 - level)

    return s.clip(lower=q_low, upper=q_high)

print("➡️ Applicazione winsorizing per SKU...")
df_filtered["Demand"] = (
    df_filtered
    .groupby(ID_COL)["Demand"]
    .transform(lambda s: winsorize_series(s, OUTLIER_LEVEL))
)
print("✔️ Winsorizing completato.")
df_filtered.head()

In [ ]:
# ======================================================================
# MODULO C — Serie Storiche
# C.1 — Costruzione delle serie storiche complete (formato TimesFM)
# ======================================================================

df_mode = df_filtered.copy()

print("➡️ Utilizzo dell'intero storico disponibile per il forecasting.")
print("SKU totali:", df_mode[ID_COL].nunique())
print("Righe totali:", len(df_mode))

sku_series = {}   # dizionario SKU → lista valori storici

for sku, group in df_mode.groupby(ID_COL):
    g = group.sort_values("Date")
    values = g["Demand"].astype(float).tolist()
    dates  = g["Date"].tolist()

    # Rimozione zeri iniziali: elimina solo gli zeri in testa alla serie
    # (prodotto non ancora lanciato). Zeri interni e finali sono mantenuti.
    if TRIM_LEADING_ZEROS:
        idx = 0
        while idx < len(values) and values[idx] == 0:
            idx += 1
        values = values[idx:]
        dates  = dates[idx:]

    if len(values) == 0:
        continue

    sku_series[sku] = values

print("Serie generate per", len(sku_series), "SKU.")
first_sku = list(sku_series.keys())[0]
print("\nEsempio prima serie:")
print("SKU:", first_sku)
print("Serie:", sku_series[first_sku][:10], "...")
print("Lunghezza:", len(sku_series[first_sku]))

In [ ]:
# ======================================================================
# MODULO C — Serie Storiche
# C.2 — Preparazione serie storiche troncate per il backtest
# ======================================================================

backtest_series = {}   # SKU → lista valori storici TRONCATI (input al modello nel backtest)
backtest_actuals = {}  # SKU → np.array degli ultimi HORIZON_BACKTEST valori reali (target)

skus_with_series = len(sku_series)
skipped_too_short = 0

print(f"➡️ Preparazione serie per backtest con HORIZON_BACKTEST = {HORIZON_BACKTEST} mesi")

for sku, values in sku_series.items():
    n = len(values)

    # Serve almeno HORIZON_BACKTEST mesi per avere dati di confronto
    if n <= HORIZON_BACKTEST:
        skipped_too_short += 1
        continue

    # Storico troncato: tutto tranne gli ultimi HORIZON_BACKTEST mesi
    # (da dare a TimesFM per simulare una previsione sul passato)
    hist_bt = values[:-HORIZON_BACKTEST]

    # Valori reali degli ultimi HORIZON_BACKTEST mesi
    # (usati come riferimento per calcolare l'accuratezza)
    act_bt = values[-HORIZON_BACKTEST:]

    # Verifica che lo storico troncato abbia ancora abbastanza punti
    if len(hist_bt) < MIN_HISTORY_POINTS:
        skipped_too_short += 1
        continue

    backtest_series[sku] = hist_bt
    backtest_actuals[sku] = np.array(act_bt, dtype=float)

print("\n📊 Riepilogo preparazione backtest:")
print(f" - SKU totali con serie storica (post-trim): {skus_with_series}")
print(f" - SKU utilizzabili per backtest:           {len(backtest_series)}")
print(f" - SKU esclusi (serie troppo corte):        {skipped_too_short}")

In [ ]:
# ======================================================================
# MODULO C — Serie Storiche
# C.3 — Funzioni di accuratezza Motul (mensile e pesata)
# ======================================================================

def accuracy_single_month(act, fcst):
    """
    Accuratezza mensile secondo la formula aziendale (Casa Madre):

        Δ = |ACT - FCST|
        ACC_i = 0 se:
            - ACT <= 0
            - FCST <= 0
            - Δ > ACT   (forecast troppo basso: meno della metà del reale)
            - Δ > FCST  (forecast troppo alto: più del doppio del reale)
        altrimenti:
            1 - Δ / ACT
    """
    act = float(act)
    fcst = float(fcst)

    # Condizioni che annullano l'accuratezza
    if act <= 0:
        return 0.0
    if fcst <= 0:
        return 0.0

    delta = abs(act - fcst)

    if delta > act:    # forecast < metà del reale
        return 0.0
    if delta > fcst:   # forecast > doppio del reale
        return 0.0

    return 1 - (delta / act)


def accuracy_weighted(act_array, fcst_array):
    """
    Accuratezza pesata per volume secondo la formula aziendale:

        ACC = Σ( ACC_i × (ACT_i + FCST_i) ) / Σ(ACT_i + FCST_i)

    Ogni mese viene pesato per il volume (reale + previsto),
    così i mesi con volumi maggiori contano di più.
    Restituisce 0 se i pesi totali sono nulli.
    """
    acc_list = []
    weights = []

    for act, fc in zip(act_array, fcst_array):
        acc_i = accuracy_single_month(act, fc)
        w_i = act + fc
        acc_list.append(acc_i * w_i)
        weights.append(w_i)

    total_w = sum(weights)
    if total_w <= 0:
        return 0.0

    return sum(acc_list) / total_w

In [ ]:
# ======================================================================
# MODULO D — Calibrazione Stagionale
# D.1 — Calcolo fattori di calibrazione (globali + per SKU)
# ======================================================================

# Se non ci sono mesi da calibrare, mappa vuota → nessun aggiustamento
if not CALIBRATION_MONTHS:
    print("ℹ️ CALIBRATION_MONTHS vuoto: nessuna calibrazione calcolata.")
    CALIBRATION_FACTORS = {}
    GLOBAL_CALIBRATION_FACTORS = {}
else:
    def month_from_label(label):
        """Estrae il numero del mese (1-12) da un'etichetta 'YYYY_MM'."""
        try:
            return int(str(label).split("_")[1])
        except Exception:
            return None

    def theil_sen_log_trend(values):
        """
        Stima robusta del trend su scala logaritmica (Theil-Sen completo).

        Calcola pendenza (beta) e intercetta (alpha) usando TUTTE le coppie
        di punti (vettorizzato con numpy). Filtra internamente gli zeri
        preservando le posizioni temporali originali, così i residui
        restano coerenti con la serie completa.

        Questa è l'unica implementazione canonica, usata sia nella
        calibrazione (Modulo D) che nel backtest (Modulo G).
        """
        y = np.array(values, dtype=float)
        valid_idx = np.where(y > 0)[0]      # indici dei valori positivi
        if len(valid_idx) < 6:
            return None, None

        y_log = np.log(y[valid_idx])         # logaritmo dei valori positivi
        t = valid_idx.astype(float)          # posizioni temporali originali

        n = len(y_log)

        # Matrice delle pendenze (triangolo superiore: j > i)
        dy = y_log[None, :] - y_log[:, None]   # dy[i,j] = y_log[j] - y_log[i]
        dt = t[None, :] - t[:, None]           # dt[i,j] = t[j] - t[i]
        mask = np.triu(np.ones((n, n), dtype=bool), k=1)
        slopes = dy[mask] / dt[mask]           # pendenze di tutte le coppie

        if len(slopes) == 0:
            return None, None

        # Mediana delle pendenze (robusto rispetto agli outlier)
        beta = float(np.median(slopes))
        # Intercetta: mediana dei residui (y_log - beta * t)
        alpha = float(np.median(y_log - beta * t))
        return alpha, beta

    # --- Costruzione tabella storica in formato wide ---
    df_hist_wide = df_filtered.pivot(
        index=ID_COL,
        columns="Period",
        values="Demand"
    ).reset_index()

    time_cols = [c for c in df_hist_wide.columns if re.match(r"^\d{4}_\d{2}$", str(c))]
    time_cols = sorted(time_cols)

    # --- Fattore globale: mediana dei residui log per mese target ---
    def global_month_medians(df_wide, time_cols, target_months):
        """Calcola la mediana dei residui log-detrendizzati per ogni mese target,
        aggregando tutti gli SKU con storico sufficiente."""
        med = {m: [] for m in target_months}

        for _, row in df_wide.iterrows():
            vals = pd.Series([row[c] for c in time_cols], index=time_cols, dtype="float")
            vals = pd.to_numeric(vals, errors="coerce")

            # Trim solo zeri iniziali, mantieni zeri interni
            nz = np.where((vals.fillna(0) > 0).values)[0]
            if len(nz) == 0:
                continue
            vals = vals.iloc[nz[0]:].dropna()

            if (vals > 0).sum() < 6:
                continue

            alpha, beta = theil_sen_log_trend(vals.values)
            if alpha is None:
                continue

            # Calcola residuo = log(reale) - trend stimato, per ogni mese target
            for i, (c, v) in enumerate(vals.items()):
                m = month_from_label(c)
                if m in med and v > 0:
                    r = np.log(v) - (alpha + beta * i)
                    med[m].append(r)

        return {
            m: (float(np.median(v)) if v else 0.0)
            for m, v in med.items()
        }

    GLOBAL_MED = global_month_medians(df_hist_wide, time_cols, CALIBRATION_MONTHS)
    GLOBAL_CALIBRATION_FACTORS = {
        m: math.exp(GLOBAL_MED[m]) for m in CALIBRATION_MONTHS
    }

    print("Fattori globali per mesi target:",
          {m: round(GLOBAL_CALIBRATION_FACTORS[m], 4) for m in CALIBRATION_MONTHS})

    # --- Fattori per-SKU ---
    CALIBRATION_FACTORS = {}  # SKU → {mese → fattore}

    hist_by_sku = df_hist_wide.set_index(ID_COL)

    for sku, row_hist in hist_by_sku.iterrows():
        vals = pd.Series([row_hist[c] for c in time_cols], index=time_cols, dtype="float")
        vals = pd.to_numeric(vals, errors="coerce")

        nz = np.where((vals.fillna(0) > 0).values)[0]
        if len(nz) == 0:
            continue

        # Trim solo zeri iniziali, mantieni zeri interni
        vals_trimmed = vals.iloc[nz[0]:].dropna()

        if (vals_trimmed > 0).sum() >= 6:
            alpha, beta = theil_sen_log_trend(vals_trimmed.values)
            if alpha is not None:
                bucket = {m: [] for m in CALIBRATION_MONTHS}
                for i, (c, v) in enumerate(vals_trimmed.items()):
                    m = month_from_label(c)
                    if m in bucket and v > 0:
                        r = np.log(v) - (alpha + beta * i)
                        bucket[m].append(r)

                per_month_med = {}
                for m in CALIBRATION_MONTHS:
                    if len(bucket[m]) >= 2:
                        # Almeno 2 osservazioni → fattore specifico per SKU
                        per_month_med[m] = float(np.median(bucket[m]))
                    else:
                        # Meno di 2 osservazioni → fallback al fattore globale
                        per_month_med[m] = GLOBAL_MED[m]
            else:
                per_month_med = {m: GLOBAL_MED[m] for m in CALIBRATION_MONTHS}
        else:
            per_month_med = {m: GLOBAL_MED[m] for m in CALIBRATION_MONTHS}

        # Converte residui log in fattori moltiplicativi: exp(mediana_residuo)
        factors = {
            m: math.exp(per_month_med[m]) for m in CALIBRATION_MONTHS
        }
        CALIBRATION_FACTORS[sku] = factors

    print("✅ Fattori di calibrazione per-SKU calcolati (bidirezionali: possono aumentare o diminuire il forecast).")

In [ ]:
# ======================================================================
# MODULO D — Calibrazione Stagionale
# D.2 — Funzione helper: fattore di calibrazione per SKU e mese
# ======================================================================

def get_calibration_factor_for_sku_month(sku, month):
    """
    Restituisce il fattore di calibrazione per uno SKU e un mese (1-12).
    Logica di priorità:
      1. Fattore specifico per-SKU (se disponibile)
      2. Fattore globale del mese (fallback)
      3. 1.0 (nessuna calibrazione)
    """
    if not CALIBRATION_MONTHS:
        return 1.0

    # Se il mese non rientra tra quelli da calibrare → nessun aggiustamento
    if month not in CALIBRATION_MONTHS:
        return 1.0

    # Priorità 1: fattore per-SKU
    if sku in CALIBRATION_FACTORS and month in CALIBRATION_FACTORS[sku]:
        return CALIBRATION_FACTORS[sku][month]

    # Priorità 2: fattore globale
    if month in GLOBAL_CALIBRATION_FACTORS:
        return GLOBAL_CALIBRATION_FACTORS[month]

    return 1.0

In [ ]:
# ======================================================================
# MODULO E — Arrotondamento
# E.1 — Funzione di arrotondamento a multiplo d'imballo
# ======================================================================

def round_to_pack(value, pack, mode=ROUNDING_MODE, decimals=ROUND_DECIMALS):
    """
    Arrotonda un valore al multiplo d'imballo (pack size) più vicino.

    Parametri:
        value:    valore da arrotondare
        pack:     multiplo d'imballo (es. 6 → arrotonda a 6, 12, 18, ...)
        mode:     "up" (per eccesso), "down" (per difetto), "nearest" (più vicino)
        decimals: numero di decimali nel risultato finale

    Se pack è nullo o <= 0, arrotonda solo ai decimali richiesti.
    """
    if pd.isna(value):
        return value
    if pack is None or pack <= 0:
        return round(float(value), decimals) if decimals is not None else float(value)

    # Dividi per il pack, arrotonda, moltiplica per il pack
    scaled = float(value) / pack

    if mode == "up":
        scaled = np.ceil(scaled)
    elif mode == "down":
        scaled = np.floor(scaled)
    else:  # nearest
        scaled = np.round(scaled)

    result = scaled * pack
    return round(result, decimals) if decimals is not None else result

In [ ]:
# ======================================================================
# MODULO F — Modello TimesFM
# F.1 — Caricamento manuale del modello (compatibile Colab/Python 3.12)
# ======================================================================

import os, sys, importlib, pathlib, re
import numpy as np

# 1. Imposta path cache Hugging Face
if COLAB:
    # Su Colab: cache locale effimera, riscarica ad ogni sessione
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
# In locale: HF_HOME non viene impostato, usa il default di huggingface_hub (~/.cache/huggingface).
# Il check ETag automatico di huggingface_hub verifica se esiste una nuova versione del modello
# ad ogni avvio, e scarica gli aggiornamenti solo se necessario.
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

MODEL_ID = "google/timesfm-2.5-200m-pytorch"

if COLAB:
    # 2. Installa dipendenze richieste dal modello (NON il pacchetto timesfm)
    !pip install -q einops huggingface_hub torch

    # 3. Scarica il repository ufficiale TimesFM (solo codice sorgente)
    if not pathlib.Path("/content/timesfm").exists():
        !git clone -q https://github.com/google-research/timesfm.git /content/timesfm

    TIMESFM_DIR = "/content/timesfm"
else:
    # In locale: le dipendenze devono essere gia' installate nell'ambiente Python.
    # Installa manualmente con: pip install einops huggingface_hub torch timesfm
    TIMESFM_DIR = "./timesfm"
    if not pathlib.Path(TIMESFM_DIR).exists():
        import subprocess
        print("📥 Clone del repository TimesFM in corso...")
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/google-research/timesfm.git",
                        TIMESFM_DIR], check=True)

# 4. Aggiungi le cartelle sorgente a sys.path per l'import manuale
SRC_DIR = TIMESFM_DIR
PKG_DIR = os.path.join(TIMESFM_DIR, "src")
T25_DIR = os.path.join(TIMESFM_DIR, "src", "timesfm", "timesfm_2p5")

if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

# 5. Import manuale del modulo PyTorch di TimesFM
# Cerca i file con "pytorch" nel nome nella cartella timesfm_2p5
torch_files = list(pathlib.Path(T25_DIR).glob("**/*pytorch*.py"))
if not torch_files:
    torch_files = list(pathlib.Path(T25_DIR).glob("**/*torch*.py"))
torch_files.sort()

if not torch_files:
    raise RuntimeError("❌ Non trovo il modulo Torch in timesfm_2p5!")

torch_mod_path = torch_files[0]
print("🧭 Modulo trovato:", torch_mod_path)

# Importa il modulo usando spec_from_file_location (bypass del pacchetto pip)
spec = importlib.util.spec_from_file_location(
    "timesfm.timesfm_2p5.timesfm_2p5_torch",
    str(torch_mod_path)
)
torch_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(torch_mod)
sys.modules["timesfm.timesfm_2p5.timesfm_2p5_torch"] = torch_mod

# 6. Identifica la classe del modello (cerca una classe con metodo "forecast")
ModelClass = None
for name, cls in vars(torch_mod).items():
    if isinstance(cls, type) and hasattr(cls, "forecast"):
        ModelClass = cls
        break

if ModelClass is None:
    raise RuntimeError("❌ Nessuna classe TimesFM con metodo forecast trovata!")

print("🧩 Classe modello:", ModelClass)

# 7. Scarica i pesi pre-addestrati da HuggingFace
# NON usa local_files_only=True, cosi' il check ETag verifica e aggiorna il modello se necessario
from huggingface_hub import hf_hub_download

print("⬇️ Download pesi (HuggingFace)…")
model = ModelClass.from_pretrained(MODEL_ID)

# 8. Configurazione parametri di inferenza
cfg = None
for mp in [
    "timesfm.config",
    "timesfm.configs",
    "timesfm.timesfm_2p5.configs.forecast_config",
]:
    try:
        m = importlib.import_module(mp)
        if hasattr(m, "ForecastConfig"):
            ForecastConfig = m.ForecastConfig
            cfg = ForecastConfig(
                max_context=512,
                max_horizon=HORIZON,
                normalize_inputs=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            )
            break
    except:
        pass

if cfg is not None:
    model.compile(cfg)
else:
    model.compile()

# 9. Rilevamento automatico CPU/GPU
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🖥️ Device: {DEVICE}")

# TimesFM non deriva da torch.nn.Module → .to() e .eval() potrebbero non essere disponibili
if hasattr(model, "to"):
    try:
        model.to(DEVICE)
        print("✔️ Modello spostato su device tramite .to()")
    except Exception:
        print("⚠️ .to() disponibile ma non utilizzabile — ignorato.")
else:
    print("ℹ️ Modello senza .to(): TimesFM gestisce automaticamente CPU/GPU.")

if hasattr(model, "eval"):
    try:
        model.eval()
        print("✔️ Modalita' eval attivata")
    except Exception:
        print("⚠️ .eval() disponibile ma non chiamabile — ignorato.")
else:
    print("ℹ️ Modello senza .eval(): TimesFM e' gia' in modalita' inferenza.")

print("✔️ Modello pronto (loader manuale TimesFM attivato).")

# 10. Test rapido di verifica (smoke test)
try:
    test = np.array([10,12,11,13,15,14,16,18,17,19], dtype=np.float32)
    pred, _ = model.forecast(horizon=3, inputs=[test])
    print("🔬 Smoke test OK →", pred[0])
except Exception as e:
    print("⚠️ Smoke test fallito:", e)

In [ ]:
# ======================================================================
# MODULO F — Modello TimesFM
# F.2 — Funzione di forecast batch per tutti gli SKU
# ======================================================================

def forecast_all_skus_point(series_dict, horizon):
    """
    Esegue il forecast point (mediana) per tutte le serie SKU in un'unica
    chiamata batch al modello. Se il batch fallisce, esegue automaticamente
    il fallback a forecast singolo per ogni SKU.

    Parametri:
        series_dict: dizionario {SKU: lista_valori_storici}
        horizon:     numero di mesi da prevedere

    Restituisce:
        results: dizionario {SKU: array_forecast}
        errors:  dizionario {SKU: messaggio_errore} per gli SKU falliti
    """
    results = {}
    errors = {}

    total = len(series_dict)
    print(f"📦 Avvio forecast point per {total} SKU (batch)...\n")

    skus = list(series_dict.keys())
    inputs = [np.array(series_dict[s], dtype=np.float32) for s in skus]

    try:
        # Tentativo batch: tutte le serie in una sola chiamata
        all_fc, _ = model.forecast(horizon=horizon, inputs=inputs)

        for i, sku in enumerate(skus):
            fc = all_fc[i]
            if hasattr(fc, "cpu"):
                fc = fc.cpu().numpy()
            results[sku] = fc

    except Exception as e:
        # Fallback: forecast singolo per ogni SKU
        print(f"⚠️ Batch fallito ({e}), fallback a forecast singolo...")
        for idx, sku in enumerate(skus):
            if VERBOSE:
                print(f"[{idx+1}/{total}] SKU {sku}... ", end="")
            try:
                point_fc, _ = model.forecast(
                    horizon=horizon,
                    inputs=[inputs[idx]]
                )
                fc = point_fc[0]
                if hasattr(fc, "cpu"):
                    fc = fc.cpu().numpy()
                results[sku] = fc
                if VERBOSE:
                    print("OK")
            except Exception as e2:
                if VERBOSE:
                    print("❌ Errore")
                errors[sku] = str(e2)

    print("\n✔️ Forecast point completato.")
    print(" - SKU riusciti:", len(results))
    print(" - SKU falliti:", len(errors))

    return results, errors

print("La funzione forecast_all_skus_point(...) è pronta all'uso.")

In [ ]:
# ======================================================================
# MODULO G — Backtest
# G.1 — Ottimizzazione scaling factor per SKU
#        (rolling-origin, griglia fine, shrinkage opzionale)
# ======================================================================
#
# Il backtest simula il forecast sul passato per trovare lo scaling factor
# ottimale per ogni SKU. Migliorie rispetto alla versione base:
#   - Rolling-origin: più origini di backtest per robustezza (N_BACKTEST_ORIGINS)
#   - Griglia fine: secondo passaggio con step 0.01 attorno al miglior q
#   - Shrinkage: miscela q per-SKU con mediana globale (SHRINKAGE_ENABLED)
#
# Pipeline per ogni origine:
#   1. Funzione di calibrazione locale (replica Modulo D senza data leakage)
#   2. Pre-indicizzazione dei dati per accesso rapido
#   3. Preparazione dati per ogni SKU (allineamento, calibrazione, pack)
#   4. Forecast batch con fallback automatico
# Poi cross-origin:
#   5. Grid search (grossolano + fine) mediato su tutte le origini
#   6. Shrinkage opzionale
#
# ======================================================================

import numpy as np
import pandas as pd
import math

# --- 1. Funzione di calibrazione stagionale locale (no data leakage) ---

def calculate_seasonality_theil_sen_local(hist_bt, dates_bt, calibration_months):
    """
    Calcola i fattori di calibrazione stagionale usando SOLO lo storico
    troncato del backtest (senza vedere i dati futuri).

    Replica la logica del Modulo D ma su dati limitati, garantendo
    assenza di data leakage. Usa theil_sen_log_trend (definita nel
    Modulo D) per coerenza con la pipeline di produzione.

    Restituisce: dizionario {mese → fattore_moltiplicativo}
    """
    if not calibration_months:
        return {}

    ts = pd.Series(hist_bt, index=dates_bt)
    ts = ts[ts > 0]

    if len(ts) < 6:
        return {}

    alpha, beta = theil_sen_log_trend(hist_bt)

    if alpha is None:
        return {}

    month_residuals = {m: [] for m in calibration_months}

    for date, val in ts.items():
        m = date.month
        if m in calibration_months:
            try:
                t = dates_bt.tolist().index(date)
                expected_log = alpha + beta * t
                actual_log = np.log(val)
                res = actual_log - expected_log
                month_residuals[m].append(res)
            except ValueError:
                continue

    factors = {}
    for m in calibration_months:
        residuals = month_residuals[m]
        if len(residuals) >= 1:
            med_res = np.median(residuals)
            factors[m] = math.exp(med_res)
        else:
            factors[m] = 1.0

    return factors

# --- 2. Pre-indicizzazione dati per SKU (ottimizza accesso ripetuto) ---

_grouped_bt = df_filtered.sort_values("Date").groupby(ID_COL)

# --- 3. Helper: valuta accuratezza per un dato scaling factor q ---

def _evaluate_q(base_fc, actuals, test_months, local_factors, pack, q):
    """
    Applica la pipeline completa (scaling → calibrazione → arrotondamento)
    e restituisce (accuratezza_pesata, peso_totale).
    """
    scale = q / 0.5
    fc_scaled = base_fc * scale

    fc_calibrated = []
    for i, val in enumerate(fc_scaled):
        if i >= len(test_months):
            break
        m = test_months[i]
        factor = local_factors.get(m, 1.0)
        fc_calibrated.append(val * factor)

    fc_calibrated = np.array(fc_calibrated, dtype=float)
    fc_rounded = np.array([
        round_to_pack(v, pack, mode=ROUNDING_MODE, decimals=ROUND_DECIMALS)
        for v in fc_calibrated
    ], dtype=float)

    acc = accuracy_weighted(actuals, fc_rounded)
    total_weight = float(np.sum(actuals + fc_rounded))
    return acc, total_weight

# --- 4. Preparazione e forecast per ogni origine di backtest ---

ORIGIN_SHIFT = 6  # mesi di distanza tra un'origine e la successiva

origins_data = []  # lista di dict {sku: {base_fc, actuals, test_months, ...}}

for origin_idx in range(N_BACKTEST_ORIGINS):
    shift = origin_idx * ORIGIN_SHIFT

    print(f"\n▶️ Origine {origin_idx + 1}/{N_BACKTEST_ORIGINS} (shift = {shift} mesi)...")

    # 4a. Preparazione dati per questa origine
    origin_prep = {}

    for sku, values in sku_series.items():
        n = len(values)

        # Serve storico sufficiente: history + backtest window + shift
        if n <= HORIZON_BACKTEST + shift:
            continue

        # Indici nella serie (già trimmed degli zeri iniziali)
        act_end = n - shift
        act_start = act_end - HORIZON_BACKTEST

        if act_start < MIN_HISTORY_POINTS:
            continue

        hist_bt = values[:act_start]
        act_bt = np.array(values[act_start:act_end], dtype=float)

        # Allineamento date dalla tabella completa
        if sku not in _grouped_bt.groups:
            continue

        sku_data = _grouped_bt.get_group(sku)
        full_dates = sku_data["Date"].tolist()
        trim_offset = len(full_dates) - n  # zeri iniziali rimossi

        hist_bt_dates = full_dates[trim_offset : trim_offset + act_start]
        act_bt_dates = full_dates[trim_offset + act_start : trim_offset + act_end]
        test_months = [d.month for d in act_bt_dates]

        # Calibrazione locale Theil-Sen (usa solo storico troncato → no leakage)
        local_factors = calculate_seasonality_theil_sen_local(
            hist_bt, pd.Series(hist_bt_dates), CALIBRATION_MONTHS
        )

        # Pack size per arrotondamento
        row_meta = sku_data[[PACK_SIZE_COL, UOM_COL]].iloc[0]
        pack_val = row_meta[PACK_SIZE_COL]
        pack = float(pack_val) if pd.notna(pack_val) and pack_val not in ("", None) else 1.0

        origin_prep[sku] = {
            'hist_np': np.array(hist_bt, dtype=np.float32),
            'actuals': act_bt,
            'test_months': test_months,
            'local_factors': local_factors,
            'pack': pack,
        }

    print(f"  SKU preparati: {len(origin_prep)}")

    if not origin_prep:
        origins_data.append({})
        continue

    # 4b. Forecast batch per questa origine
    bt_skus = list(origin_prep.keys())
    bt_inputs = [origin_prep[s]['hist_np'] for s in bt_skus]

    try:
        all_fc, _ = model.forecast(horizon=HORIZON_BACKTEST, inputs=bt_inputs)
        for i, sku in enumerate(bt_skus):
            fc = all_fc[i]
            if hasattr(fc, "cpu"):
                fc = fc.cpu().numpy()
            origin_prep[sku]['base_fc'] = np.array(fc, dtype=float)
    except Exception as _e:
        print(f"  ⚠️ Batch fallito ({_e}), fallback a forecast singolo...")
        for sku in bt_skus:
            try:
                _fc, _ = model.forecast(
                    horizon=HORIZON_BACKTEST,
                    inputs=[origin_prep[sku]['hist_np']]
                )
                _fc = _fc[0]
                if hasattr(_fc, "cpu"):
                    _fc = _fc.cpu().numpy()
                origin_prep[sku]['base_fc'] = np.array(_fc, dtype=float)
            except Exception:
                origin_prep[sku]['base_fc'] = None

    # Rimuovi SKU senza forecast valido
    origin_prep = {s: p for s, p in origin_prep.items() if p.get('base_fc') is not None}
    print(f"  SKU con forecast valido: {len(origin_prep)}")

    origins_data.append(origin_prep)

# --- 5. Grid search cross-origin (griglia grossolana + raffinamento fine) ---

# Tutti gli SKU presenti in almeno un'origine
all_bt_skus = set()
for od in origins_data:
    all_bt_skus.update(od.keys())

print(f"\n▶️ Grid search su {len(all_bt_skus)} SKU "
      f"({N_BACKTEST_ORIGINS} origini, griglia fine attiva)...")

results_list = []

for idx, sku in enumerate(sorted(all_bt_skus), start=1):

    # --- Passo 1: Griglia grossolana (step 0.05) ---
    coarse_acc = {}
    for q in QUANTILE_GRID:
        accs = []
        for od in origins_data:
            if sku in od:
                acc, _ = _evaluate_q(
                    od[sku]['base_fc'], od[sku]['actuals'],
                    od[sku]['test_months'], od[sku]['local_factors'],
                    od[sku]['pack'], q
                )
                accs.append(acc)
        if accs:
            coarse_acc[q] = np.mean(accs)

    if not coarse_acc:
        continue

    best_q_coarse = max(coarse_acc, key=lambda x: coarse_acc[x])

    # --- Passo 2: Griglia fine (step 0.01 attorno al miglior q grossolano) ---
    fine_candidates = [round(best_q_coarse + d, 2)
                       for d in np.arange(-0.04, 0.05, 0.01)]
    fine_candidates = [q for q in fine_candidates
                       if 0.05 <= q <= 0.95 and q not in coarse_acc]

    fine_acc = {}
    for q in fine_candidates:
        accs = []
        for od in origins_data:
            if sku in od:
                acc, _ = _evaluate_q(
                    od[sku]['base_fc'], od[sku]['actuals'],
                    od[sku]['test_months'], od[sku]['local_factors'],
                    od[sku]['pack'], q
                )
                accs.append(acc)
        if accs:
            fine_acc[q] = np.mean(accs)

    # Unione delle due griglie: seleziona il miglior q globale
    all_acc = {**coarse_acc, **fine_acc}
    best_q = max(all_acc, key=lambda x: all_acc[x])
    best_accuracy = all_acc[best_q]

    # Peso totale dall'origine primaria (per il KPI globale pesato)
    if sku in origins_data[0]:
        _, total_weight = _evaluate_q(
            origins_data[0][sku]['base_fc'], origins_data[0][sku]['actuals'],
            origins_data[0][sku]['test_months'], origins_data[0][sku]['local_factors'],
            origins_data[0][sku]['pack'], best_q
        )
    else:
        total_weight = 0.0
        for od in origins_data:
            if sku in od:
                _, total_weight = _evaluate_q(
                    od[sku]['base_fc'], od[sku]['actuals'],
                    od[sku]['test_months'], od[sku]['local_factors'],
                    od[sku]['pack'], best_q
                )
                break

    results_list.append({
        "SKU": sku,
        "BestQuantile": best_q,
        "BestAccuracy": best_accuracy,
        "TotalWeight": total_weight,
    })

    if VERBOSE and idx % 10 == 0:
        print(f"  [{idx}] {sku} -> BestQ: {best_q}, Acc: {best_accuracy:.2f}")

# --- 6. Shrinkage opzionale ---

if SHRINKAGE_ENABLED and results_list:
    print("\n➡️ Applicazione shrinkage dello scaling factor...")

    q_values = [r["BestQuantile"] for r in results_list]
    q_global = float(np.median(q_values))

    for r in results_list:
        sku = r["SKU"]
        hist_len = len(sku_series.get(sku, []))
        # alpha cresce con lo storico: piena fiducia a 36 mesi
        alpha = min(1.0, hist_len / 36.0)
        q_orig = r["BestQuantile"]
        q_shrunk = round(alpha * q_orig + (1 - alpha) * q_global, 2)
        q_shrunk = max(0.05, min(0.95, q_shrunk))
        r["BestQuantile"] = q_shrunk

        # Ricalcola accuratezza e peso al q shrinkato
        accs = []
        for od in origins_data:
            if sku in od:
                acc, _ = _evaluate_q(
                    od[sku]['base_fc'], od[sku]['actuals'],
                    od[sku]['test_months'], od[sku]['local_factors'],
                    od[sku]['pack'], q_shrunk
                )
                accs.append(acc)
        if accs:
            r["BestAccuracy"] = np.mean(accs)

        if sku in origins_data[0]:
            _, tw = _evaluate_q(
                origins_data[0][sku]['base_fc'], origins_data[0][sku]['actuals'],
                origins_data[0][sku]['test_months'], origins_data[0][sku]['local_factors'],
                origins_data[0][sku]['pack'], q_shrunk
            )
            r["TotalWeight"] = tw

    print(f"  q globale (mediana): {q_global:.2f}")
    print(f"  Shrinkage applicato a {len(results_list)} SKU")

# --- 7. Riepilogo risultati ---

df_backtest_results = pd.DataFrame(results_list)
print("\n✔️ Backtest completato.")

if not df_backtest_results.empty:
    mean_acc_simple = df_backtest_results["BestAccuracy"].mean()

    total_w = df_backtest_results["TotalWeight"].sum()
    if total_w > 0:
        weighted_acc_global = (df_backtest_results["BestAccuracy"] * df_backtest_results["TotalWeight"]).sum() / total_w
    else:
        weighted_acc_global = 0.0

    print(f"\n📊 RISULTATI BACKTEST:")
    print(f" - SKU Processati: {len(df_backtest_results)}")
    print(f" - Origini backtest: {N_BACKTEST_ORIGINS}")
    print(f" - Griglia: grossolana (step 0.05) + fine (step 0.01)")
    if SHRINKAGE_ENABLED:
        print(f" - Shrinkage: attivo (q globale = {q_global:.2f})")
    else:
        print(f" - Shrinkage: disattivato")
    print(f" - Media Semplice (peso uguale per SKU):  {mean_acc_simple:.2%}")
    print(f" - MEDIA PESATA GLOBALE (peso per volume): {weighted_acc_global:.2%}  <-- KPI REALE")

    display(df_backtest_results.head())
else:
    print("⚠️ Nessun risultato generato.")

In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.1 — Esecuzione forecast point TimesFM su storico completo
# ======================================================================

print("➡️ Avvio forecast (TimesFM)")
print(f"Orizzonte scelto: {HORIZON} mesi\n")

fc_results, fc_errors = forecast_all_skus_point(
    series_dict=sku_series,
    horizon=HORIZON
)

print("\n📊 Riepilogo forecast:")
print(" - SKU con forecast:", len(fc_results))
print(" - SKU con errori:", len(fc_errors))

if fc_errors:
    print("\n❌ Errori rilevati su:")
    for sku, err in fc_errors.items():
        if VERBOSE:
            print(f"   SKU {sku}: {err}")
else:
    print("✔️ Nessun errore nei forecast.")

In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.2 — Applicazione scaling ottimale + calibrazione + arrotondamento
# ======================================================================

fc_rows = []

print("➡️ Ricostruzione forecast per-SKU su timeline globale (pipeline completa)...")

# Timeline futura: parte dal mese successivo all'ultima data storica
last_date_global = df_filtered["Date"].max()
print("Ultima data storica globale:", last_date_global)

start_date_global = last_date_global + pd.offsets.MonthBegin(1)
future_dates_global = [
    start_date_global + pd.offsets.MonthBegin(i)
    for i in range(HORIZON)
]

# Mappa SKU → scaling factor ottimale (dal backtest)
best_q_map = {}
if "df_backtest_results" in globals():
    best_q_map = dict(zip(df_backtest_results["SKU"], df_backtest_results["BestQuantile"]))
else:
    print("⚠️ Attenzione: df_backtest_results non trovato, uso q=0.5 per tutti gli SKU.")

def get_best_quantile_for_sku(sku):
    """Restituisce lo scaling factor ottimale per lo SKU, o 0.5 (mediana) se non disponibile."""
    q = best_q_map.get(sku, 0.5)
    try:
        return float(q)
    except Exception:
        return 0.5

# Mappa SKU → pack size (per arrotondamento)
meta_pack = (
    df_filtered[[ID_COL, PACK_SIZE_COL, UOM_COL]]
    .drop_duplicates()
    .set_index(ID_COL)
)

def get_pack_for_sku(sku):
    """Restituisce il pack size per lo SKU, default 1.0 se mancante o non numerico."""
    if sku not in meta_pack.index:
        return 1.0
    v = meta_pack.loc[sku, PACK_SIZE_COL]
    if pd.isna(v) or v in ("", None):
        return 1.0
    try:
        return float(v)
    except Exception:
        return 1.0

# Pipeline completa per ogni SKU: scaling → calibrazione → arrotondamento
for sku, base_fc in fc_results.items():
    base_fc = np.array(base_fc, dtype=float)

    # 1. Scaling factor ottimale (dal backtest, o 0.5 come default)
    q = get_best_quantile_for_sku(sku)
    scale = q / 0.5  # fattore moltiplicativo rispetto alla mediana

    # 2. Applica scaling al forecast base
    fc_scaled = base_fc * scale

    # 3. Pack size per arrotondamento
    pack = get_pack_for_sku(sku)

    # Allinea lunghezza forecast e date future
    steps = min(len(fc_scaled), len(future_dates_global))

    for i in range(steps):
        date = future_dates_global[i]
        month = int(date.month)

        # 4. Calibrazione stagionale (stessa logica del backtest)
        factor = get_calibration_factor_for_sku_month(sku, month)
        fc_calibrated = fc_scaled[i] * factor

        # 5. Arrotondamento al multiplo d'imballo
        fc_final = round_to_pack(
            value=fc_calibrated,
            pack=pack,
            mode=ROUNDING_MODE,
            decimals=ROUND_DECIMALS
        )

        fc_rows.append({
            ID_COL: sku,
            "Date": date,
            "Period": date.strftime("%Y_%m"),
            "Forecast": float(fc_final),      # valore finale (scalato + calibrato + arrotondato)
            "BestQuantile": q                 # per tracciabilità
        })

df_fc_long = pd.DataFrame(fc_rows)

print("Anteprima forecast LONG (valori finali: scalato + calibrato + arrotondato):")
df_fc_long.head()

In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.3 — Conversione forecast da formato LONG a WIDE
# ======================================================================

print("➡️ Conversione forecast LONG → WIDE (valori finali)...")

# Pivot: da una riga per SKU/mese a una riga per SKU con colonne per mese
df_fc_wide = df_fc_long.pivot(
    index=ID_COL,
    columns="Period",
    values="Forecast"   # valori già finali (scalati + calibrati + arrotondati)
).reset_index()

# Ordina le colonne forecast cronologicamente
forecast_cols = sorted([c for c in df_fc_wide.columns if re.match(r"^\d{4}_\d{2}$", str(c))])
ordered_cols = [ID_COL] + forecast_cols
df_fc_wide = df_fc_wide[ordered_cols]

print("Anteprima forecast WIDE (finale):")
df_fc_wide.head()

In [ ]:
# ======================================================================
# MODULO I — Pianificazione Inventario (ABC, XYZ, Scorta di Sicurezza)
# I.1 — Classificazione ABC/XYZ e calcolo scorta di sicurezza
# ======================================================================

from scipy.stats import norm

def calculate_inventory_logic(df_history, meta_df):
    """
    Calcola la classificazione ABC/XYZ e la scorta di sicurezza per ogni SKU.

    Fasi:
      1. Preparazione dati (ultimi SS_LOOKBACK_MONTHS mesi di storico)
      2. Classificazione ABC (Pareto sui volumi cumulativi)
      3. Classificazione XYZ (coefficiente di variazione della domanda)
      4. Classe combinata ABC/XYZ → livello di servizio target
      5. Calcolo scorta di sicurezza: Z × σ × √((LT + ReorderPeriod) / 30)
         arrotondata PER ECCESSO al multiplo d'imballo
    """
    print("➡️ Avvio calcolo ABC / XYZ / Scorta di Sicurezza...")

    # --- 1. Preparazione dati (ultimi N mesi di storico winsorizzato) ---
    last_date = df_history["Date"].max()
    start_date_ss = last_date - pd.DateOffset(months=SS_LOOKBACK_MONTHS-1)

    df_ss = df_history[df_history["Date"] >= start_date_ss].copy()

    # Statistiche per SKU: volume totale, media e deviazione standard
    stats = df_ss.groupby(ID_COL)["Demand"].agg(["sum", "mean", "std"]).reset_index()
    stats.rename(columns={"sum": "TotalVol", "mean": "AvgDemand", "std": "StdDev"}, inplace=True)
    stats["StdDev"] = stats["StdDev"].fillna(0)

    # --- 2. Classificazione ABC (Pareto sui volumi) ---
    stats = stats.sort_values("TotalVol", ascending=False)
    total_volume_global = stats["TotalVol"].sum()

    if total_volume_global <= 0:
        # Guardia: se nessun volume nel lookback, tutti gli SKU → classe C
        stats["ABC"] = "C"
    else:
        stats["CumVol"] = stats["TotalVol"].cumsum()
        stats["CumPerc"] = stats["CumVol"] / total_volume_global

        def get_abc(p):
            if p <= ABC_LIMITS["A"]: return "A"
            elif p <= ABC_LIMITS["B"]: return "B"
            return "C"

        stats["ABC"] = stats["CumPerc"].apply(get_abc)

    # --- 3. Classificazione XYZ (volatilità della domanda) ---
    stats["CV"] = stats["StdDev"] / stats["AvgDemand"]
    stats["CV"] = stats["CV"].replace([np.inf, -np.inf], 999.0).fillna(0)

    def get_xyz(cv):
        if cv <= XYZ_LIMITS["X"]: return "X"
        elif cv <= XYZ_LIMITS["Y"]: return "Y"
        return "Z"

    stats["XYZ"] = stats["CV"].apply(get_xyz)

    # --- 4. Classe combinata (es. "AX", "BZ", "CY") ---
    stats["Class"] = stats["ABC"] + stats["XYZ"]

    # --- 5. Recupero lead time e pack size ---
    if LT_COL_NAME in meta_df.columns:
        lt_map = meta_df[[ID_COL, LT_COL_NAME]].drop_duplicates(subset=ID_COL)
        stats = stats.merge(lt_map, on=ID_COL, how="left")
        stats["LT_Final"] = stats[LT_COL_NAME].fillna(DEFAULT_LEAD_TIME)
    else:
        stats["LT_Final"] = DEFAULT_LEAD_TIME

    if PACK_SIZE_COL in meta_df.columns:
        pack_map = meta_df[[ID_COL, PACK_SIZE_COL]].drop_duplicates(subset=ID_COL)
        stats = stats.merge(pack_map, on=ID_COL, how="left")
    else:
        stats[PACK_SIZE_COL] = 1.0

    # --- 6. Calcolo scorta di sicurezza ---
    # Formula: SS = Z_score × σ × √((LT + ReorderPeriod) / 30)
    # Arrotondata PER ECCESSO al multiplo d'imballo (indipendentemente da ROUNDING_MODE)
    ss_values = []

    for _, row in stats.iterrows():
        cls = row["Class"]
        sigma = row["StdDev"]
        lt = row["LT_Final"]

        pack_val = row.get(PACK_SIZE_COL, 1.0)
        try:
            pack = float(pack_val) if pd.notna(pack_val) and pack_val > 0 else 1.0
        except:
            pack = 1.0

        # Livello di servizio target dalla matrice ABC/XYZ
        sl_target = SERVICE_LEVEL_MATRIX.get(cls, 0.0)

        if sl_target <= 0 or sigma <= 0:
            ss_values.append(0.0)
            continue

        z_score = norm.ppf(sl_target)          # Z dalla distribuzione normale
        time_factor = np.sqrt((lt + REORDER_PERIOD) / 30.0)  # fattore temporale

        ss_raw = z_score * sigma * time_factor
        # Arrotondamento SEMPRE per eccesso al multiplo d'imballo
        ss_final = round_to_pack(ss_raw, pack, mode="up", decimals=ROUND_DECIMALS)

        ss_values.append(ss_final)

    stats["SafetyStock"] = ss_values

    cols_to_keep = [ID_COL, "LT_Final", "ABC", "XYZ", "SafetyStock"]
    return stats[cols_to_keep]

if CALCULATE_SS:
    df_inventory = calculate_inventory_logic(df_filtered, df_raw)
    print("\nEsempio classificazione e scorta di sicurezza:")
    display(df_inventory.head())
else:
    df_inventory = None

In [ ]:
# ======================================================================
# MODULO J — Output Finale
# J.1 — Costruzione tabella finale e export Excel
# ======================================================================

print("➡️ Costruzione tabella finale (Metadati + Inventario + Storico + Forecast)...")

# 1. Metadati base (SKU, descrizione, pack, unita' di misura)
meta = df_filtered[[ID_COL, DESC_COL, PACK_SIZE_COL, UOM_COL]].drop_duplicates()

# 2. Dati inventario (classificazione ABC/XYZ + scorta di sicurezza)
if df_inventory is not None:
    df_inv_clean = df_inventory.rename(columns={"LT_Final": "LT"})
    out_step1 = meta.merge(df_inv_clean, on=ID_COL, how="left")
else:
    out_step1 = meta.copy()

# 3. Storico in formato wide (una colonna per mese)
df_hist_wide = df_filtered.pivot(
    index=ID_COL,
    columns="Period",
    values="Demand"
).reset_index()

# 4. Forecast in formato wide (colonne con prefisso "f" per distinguerle dallo storico)
df_fc_pref = df_fc_wide.copy()
df_fc_pref.rename(
    columns={c: f"f{c}" for c in df_fc_pref.columns if c != ID_COL},
    inplace=True
)

# 5. Merge finale: metadati + inventario + storico + forecast
out_final = out_step1.merge(df_hist_wide, on=ID_COL, how="left")
out_final = out_final.merge(df_fc_pref, on=ID_COL, how="left")

# Forza SKU come stringa per evitare problemi di formattazione in Excel
out_final[ID_COL] = out_final[ID_COL].astype(str)

# Riempie eventuali NaN nella scorta di sicurezza con 0
if "SafetyStock" in out_final.columns:
    out_final["SafetyStock"] = out_final["SafetyStock"].fillna(0)

print(f"✔️ Tabella finale pronta. Shape: {out_final.shape}")

# Export Excel
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_BASE + OUTPUT_SUFFIX + ".xlsx")
out_final.to_excel(output_path, index=False)
print(f"✔️ File salvato in {output_path}")

if COLAB:
    # Modalita' Colab: download automatico nel browser
    files.download(output_path)
else:
    # Modalita' locale: salvataggio nella cartella di output
    if ASK_SAVE_PATH:
        # Apre finestra di dialogo per scegliere dove salvare una copia
        import shutil
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)  # forza il dialog in primo piano su Windows
        save_path = filedialog.asksaveasfilename(
            title="Salva il file Excel di output",
            defaultextension=".xlsx",
            initialfile=OUTPUT_FILE_BASE + OUTPUT_SUFFIX + ".xlsx",
            filetypes=[("Excel files", "*.xlsx")],
            parent=root
        )
        root.destroy()
        if save_path:
            shutil.copy2(output_path, save_path)
            print(f"✔️ Copia salvata in: {save_path}")
        else:
            print(f"ℹ️ Salvataggio annullato. Il file resta in: {os.path.abspath(output_path)}")
    else:
        print(f"📁 File di output: {os.path.abspath(output_path)}")

# === RIEPILOGO TEMPI DI ESECUZIONE ===
# Impedisci a _timer_stop di stampare per questa cella (il tempo e' incluso nel riepilogo)
_suppress_timer_print = True

# Stima il tempo della cella corrente (il callback _timer_stop non e' ancora scattato)
_this_cell_elapsed = _time.time() - _cell_start.get('t', _time.time())

if _cell_times or _this_cell_elapsed > 0:
    # Numero della cella corrente: la successiva rispetto all'ultima registrata
    _this_cell_num = max(_cell_times.keys()) + 1 if _cell_times else 1
    totale = sum(_cell_times.values()) + _this_cell_elapsed
    minuti_tot, secondi_tot = divmod(totale, 60)
    print(f"\n{'='*50}")
    print(f"\u23f1\ufe0f RIEPILOGO TEMPI DI ESECUZIONE")
    print(f"{'='*50}")
    for cella, t in sorted(_cell_times.items()):
        m, s = divmod(t, 60)
        if m >= 1:
            print(f"  Cella [{cella:>2}]: {int(m)}m {s:>5.1f}s")
        else:
            print(f"  Cella [{cella:>2}]:    {s:>5.1f}s")
    # Cella corrente (riepilogo + export)
    _m, _s = divmod(_this_cell_elapsed, 60)
    if _m >= 1:
        print(f"  Cella [{_this_cell_num:>2}]: {int(_m)}m {_s:>5.1f}s")
    else:
        print(f"  Cella [{_this_cell_num:>2}]:    {_s:>5.1f}s")
    print(f"{chr(9472)*50}")
    if minuti_tot >= 1:
        print(f"  TOTALE:     {int(minuti_tot)}m {secondi_tot:.1f}s")
    else:
        print(f"  TOTALE:     {secondi_tot:.1f}s")
    print(f"{'='*50}")